# PaperMind AI — Transformer fine-tuning (Colab T4)

**ML task:** *question → passage semantic retrieval* (bi-encoder). Given a natural-language question, rank the passages of a document so that the one containing the answer comes first.

**Model:** `sentence-transformers/all-MiniLM-L6-v2` (6-layer transformer encoder, ~22.7M parameters) fine-tuned with **Multiple-Negatives-Ranking loss** (in-batch contrastive learning).

**Data:** SQuAD v1.1 (Wikipedia) + SciQ (science-exam passages). Training uses only *train* splits; a dev split calibrates the evidence threshold; the official SQuAD validation split and SciQ test split are the **held-out test sets**.

**Baselines:** TF-IDF, BM25, and the *pretrained, not fine-tuned* MiniLM.

**Outputs:** a model folder `papermind-minilm/` (goes into `backend/models/`) and a `reports/` folder (metrics, plots) for your CA2 write-up.

> Before running: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Whole notebook takes roughly 15–25 minutes.

## 1. Check GPU and install libraries

In [ ]:
!nvidia-smi
!pip install -q -U "sentence-transformers>=3.0" datasets scikit-learn matplotlib pandas

## 2. Configuration (everything you may want to change is here)

In [ ]:
import random, json, time, os, platform
import numpy as np, torch

SEED = 42
BASE_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
USE_SCIQ     = True      # add SciQ (scientific text) to train/dev/test
EPOCHS       = 2
BATCH_SIZE   = 64        # larger batch = more in-batch negatives (64 fits a T4 easily)
LR           = 3e-5
MAX_SEQ_LEN  = 256       # ~190 words; the app's passages are ~140 words
MAX_TRAIN_PAIRS = None   # e.g. 20000 for a quick smoke test; None = use everything
OUT_DIR      = "/content/outputs"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Load and split the data

In [ ]:
from datasets import load_dataset

squad = load_dataset("rajpurkar/squad")
squad_rows = lambda split: [dict(q=r["question"].strip(), c=r["context"].strip(),
                                 group=r["title"], src="squad") for r in split]
squad_all, squad_test = squad_rows(squad["train"]), squad_rows(squad["validation"])

# split SQuAD train -> train/dev BY ARTICLE so no passage leaks between them
titles = sorted({r["group"] for r in squad_all})
random.Random(SEED).shuffle(titles)
dev_titles = set(titles[: int(0.1 * len(titles))])
squad_train = [r for r in squad_all if r["group"] not in dev_titles]
squad_dev   = [r for r in squad_all if r["group"] in dev_titles]

sciq_train, sciq_dev, sciq_test = [], [], []
if USE_SCIQ:
    sciq = load_dataset("allenai/sciq")
    def sciq_rows(split):
        return [dict(q=r["question"].strip(), c=r["support"].strip(), group=r["support"][:80], src="sciq")
                for r in split if len(r["support"].strip()) > 80]
    sciq_train, sciq_dev, sciq_test = sciq_rows(sciq["train"]), sciq_rows(sciq["validation"]), sciq_rows(sciq["test"])

train_rows = squad_train + sciq_train
dev_rows   = squad_dev + sciq_dev
random.Random(SEED).shuffle(train_rows)
if MAX_TRAIN_PAIRS: train_rows = train_rows[:MAX_TRAIN_PAIRS]

split_sizes = {"train_pairs": len(train_rows), "dev_pairs": len(dev_rows),
               "test_squad": len(squad_test), "test_sciq": len(sciq_test)}
print(split_sizes)
print("\nExample pair:\nQ:", train_rows[0]["q"], "\nP:", train_rows[0]["c"][:300], "...")

## 4. Evaluation helpers (metrics: Recall@k, MRR@10, nDCG@10)

In [ ]:
# ---- Retrieval evaluation helpers (pure numpy / scikit-learn) ----
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import roc_auc_score

def make_eval(rows):
    """rows -> (queries, corpus, gold_index). Corpus = unique contexts."""
    corpus = sorted({r["c"] for r in rows})
    idx = {c: i for i, c in enumerate(corpus)}
    return [r["q"] for r in rows], corpus, np.array([idx[r["c"]] for r in rows])

def rank_metrics(scores, gold, ks=(1, 3, 5, 10)):
    """scores: (n_queries, n_passages). Pessimistic tie-breaking (ties count against us)."""
    g = scores[np.arange(len(gold)), gold]
    rank = (scores >= g[:, None]).sum(1)          # 1 = gold passage ranked first
    out = {f"Recall@{k}": float((rank <= k).mean()) for k in ks}
    out["MRR@10"] = float(np.where(rank <= 10, 1.0 / rank, 0.0).mean())
    out["nDCG@10"] = float(np.where(rank <= 10, 1.0 / np.log2(rank + 1), 0.0).mean())
    return out

def tfidf_scores(queries, corpus):
    vec = TfidfVectorizer(stop_words="english", sublinear_tf=True)
    D = vec.fit_transform(corpus)
    return (vec.transform(queries) @ D.T).toarray()

def bm25_scores(queries, corpus, k1=1.5, b=0.75):
    cv = CountVectorizer(stop_words="english")
    tf = cv.fit_transform(corpus).astype(np.float32).tocoo()
    n_docs = tf.shape[0]
    df = np.bincount(tf.col, minlength=tf.shape[1])
    idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5))
    dl = np.asarray(tf.sum(1)).ravel()
    denom = tf.data + k1 * (1 - b + b * dl[tf.row] / dl.mean())
    W = sparse.csr_matrix((tf.data * (k1 + 1) / denom * idf[tf.col], (tf.row, tf.col)), shape=tf.shape)
    Q = (cv.transform(queries) > 0).astype(np.float32)
    return (Q @ W.T).toarray()

def embed_scores(model, queries, corpus):
    enc = lambda t: model.encode(t, batch_size=128, convert_to_numpy=True,
                                 normalize_embeddings=True, show_progress_bar=False)
    return enc(queries) @ enc(corpus).T

# ---- Evidence-sufficiency ("hallucination control") helpers ----
def pair_scores(model, queries, corpus, gold, seed=0):
    """Cosine of each question with (a) its gold passage and (b) a random OTHER passage."""
    enc = lambda t: model.encode(t, batch_size=128, convert_to_numpy=True,
                                 normalize_embeddings=True, show_progress_bar=False)
    Q, C = enc(queries), enc(corpus)
    rng = np.random.default_rng(seed)
    neg = rng.integers(0, len(corpus), size=len(gold))
    neg = np.where(neg == gold, (neg + 1) % len(corpus), neg)
    return (Q * C[gold]).sum(1), (Q * C[neg]).sum(1)

def best_threshold(pos, neg):
    """Threshold maximising F1 for 'passage answers the question' on a balanced dev set."""
    cands = np.linspace(-0.2, 0.9, 221)
    best = (0.0, 0.0)
    for t in cands:
        tp, fn = (pos >= t).sum(), (pos < t).sum()
        fp = (neg >= t).sum()
        f1 = 2 * tp / max(2 * tp + fp + fn, 1)
        if f1 > best[0]:
            best = (f1, float(t))
    return best[1]

def threshold_report(pos, neg, t):
    tp, fn, fp, tn = (pos >= t).sum(), (pos < t).sum(), (neg >= t).sum(), (neg < t).sum()
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"threshold": round(float(t), 4),
            "accuracy": float((tp + tn) / (tp + tn + fp + fn)),
            "precision": float(prec), "recall": float(rec),
            "f1": float(2 * prec * rec / max(prec + rec, 1e-9)),
            "auc": float(roc_auc_score(np.r_[np.ones_like(pos), np.zeros_like(neg)], np.r_[pos, neg]))}


## 5. Baselines: TF-IDF, BM25 and the *pretrained* (not yet fine-tuned) MiniLM

In [ ]:
from sentence_transformers import SentenceTransformer

test_sets = {"SQuAD-test": squad_test}
if USE_SCIQ: test_sets["SciQ-test"] = sciq_test
evals = {name: make_eval(rows) for name, rows in test_sets.items()}

results = {}                       # results[model][test_set] = metrics dict
def record(model_name, fn):
    results[model_name] = {}
    for ds, (q, corpus, gold) in evals.items():
        results[model_name][ds] = rank_metrics(fn(q, corpus), gold)
    print(model_name, {ds: round(m["MRR@10"], 4) for ds, m in results[model_name].items()})

record("TF-IDF", tfidf_scores)
record("BM25", bm25_scores)
baseline = SentenceTransformer(BASE_MODEL, device="cuda"); baseline.max_seq_length = MAX_SEQ_LEN
record("MiniLM (pretrained)", lambda q, c: embed_scores(baseline, q, c))

## 6. Fine-tune the transformer

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sentence_transformers.training_args import BatchSamplers

model = SentenceTransformer(BASE_MODEL, device="cuda")
model.max_seq_length = MAX_SEQ_LEN
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}  |  architecture:\n{model}")

train_ds = Dataset.from_dict({"anchor": [r["q"] for r in train_rows],
                              "positive": [r["c"] for r in train_rows]})
loss = losses.MultipleNegativesRankingLoss(model, scale=20.0)

args = SentenceTransformerTrainingArguments(
    output_dir="/content/st_ckpt",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    fp16=True,                                   # T4 supports fp16 (not bf16)
    batch_sampler=BatchSamplers.NO_DUPLICATES,   # same passage never appears twice in a batch (no false negatives)
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_ds, loss=loss)
t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
print(f"Training finished in {train_minutes:.1f} min")

## 7. Evaluate the fine-tuned model on the held-out test sets

In [ ]:
model.eval()
record("MiniLM (fine-tuned)", lambda q, c: embed_scores(model, q, c))

import pandas as pd
rows = []
for m, per_ds in results.items():
    for ds, met in per_ds.items():
        rows.append({"Model": m, "Test set": ds, **{k: round(v, 4) for k, v in met.items()}})
df = pd.DataFrame(rows)
display(df)

## 8. Evidence-sufficiency calibration (hallucination control)
The app refuses to answer when the best passage's cosine similarity is below a threshold. We pick that threshold on the **dev** set (question + its true passage = *answerable*; question + a random other passage = *no evidence*) and report how well it works on the **test** sets.

In [ ]:
dev_q, dev_corpus, dev_gold = make_eval(dev_rows)
results_threshold = {}
chosen = {}
for name, m in [("MiniLM (pretrained)", baseline), ("MiniLM (fine-tuned)", model)]:
    pos, neg = pair_scores(m, dev_q, dev_corpus, dev_gold, seed=SEED)
    t = best_threshold(pos, neg)
    chosen[name] = t
    results_threshold[name] = {}
    for ds, (q, corpus, gold) in evals.items():
        p, n = pair_scores(m, q, corpus, gold, seed=SEED)
        results_threshold[name][ds] = threshold_report(p, n, t)
    print(name, "-> dev-chosen threshold:", round(t, 3))
display(pd.DataFrame([{"Model": m, "Test set": ds, **r} for m, d in results_threshold.items() for ds, r in d.items()]).round(4))

THRESHOLD = chosen["MiniLM (fine-tuned)"]

## 9. Plots for the report

In [ ]:
import matplotlib  # noqa
import matplotlib.pyplot as plt
os.makedirs(f"{OUT_DIR}/reports", exist_ok=True)

# (a) retrieval comparison
fig, axes = plt.subplots(1, len(evals), figsize=(6 * len(evals), 4), squeeze=False)
order = ["TF-IDF", "BM25", "MiniLM (pretrained)", "MiniLM (fine-tuned)"]
for ax, ds in zip(axes[0], evals):
    w, ks = 0.2, ["Recall@1", "Recall@5", "MRR@10", "nDCG@10"]
    for i, m in enumerate(order):
        ax.bar(np.arange(len(ks)) + i * w, [results[m][ds][k] for k in ks], w, label=m)
    ax.set_xticks(np.arange(len(ks)) + 1.5 * w); ax.set_xticklabels(ks)
    ax.set_ylim(0, 1); ax.set_title(f"Passage retrieval on {ds}"); ax.grid(axis="y", alpha=.3)
axes[0][0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/reports/retrieval_comparison.png", dpi=160); plt.show()

# (b) training loss
logs = [(l["step"], l["loss"]) for l in trainer.state.log_history if "loss" in l]
plt.figure(figsize=(6, 3.5)); plt.plot(*zip(*logs))
plt.xlabel("step"); plt.ylabel("MNR loss"); plt.title("Training loss"); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/reports/training_loss.png", dpi=160); plt.show()

# (c) score distributions + threshold
pos, neg = pair_scores(model, dev_q, dev_corpus, dev_gold, seed=SEED)
plt.figure(figsize=(6, 3.5))
plt.hist(neg, bins=60, alpha=.6, label="no evidence (random passage)"); plt.hist(pos, bins=60, alpha=.6, label="gold passage")
plt.axvline(THRESHOLD, color="k", ls="--", label=f"threshold={THRESHOLD:.2f}")
plt.xlabel("cosine similarity"); plt.legend(); plt.title("Evidence threshold (dev set, fine-tuned)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/reports/threshold_distribution.png", dpi=160); plt.show()

## 10. Save the model and a reproducibility report

In [ ]:
import sentence_transformers, transformers, datasets as hf_datasets, shutil
MODEL_OUT = f"{OUT_DIR}/papermind-minilm"
model.save(MODEL_OUT)

report = {
    "task": "question -> passage semantic retrieval (bi-encoder), + evidence-sufficiency threshold",
    "base_model": BASE_MODEL,
    "parameters": int(n_params),
    "architecture": "6-layer transformer encoder (hidden 384, 12 heads) + mean pooling + L2 normalisation",
    "loss": "MultipleNegativesRankingLoss (scale=20), in-batch negatives, NO_DUPLICATES batch sampler",
    "training": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LR, "warmup_ratio": 0.1,
                 "max_seq_length": MAX_SEQ_LEN, "fp16": True, "seed": SEED,
                 "optimizer": "AdamW (HF Trainer default)", "train_minutes": round(train_minutes, 2),
                 "gpu": torch.cuda.get_device_name(0)},
    "datasets": {"SQuAD v1.1 (rajpurkar/squad)": "train split -> train/dev by article (90/10); validation split = test",
                 "SciQ (allenai/sciq)": "train / validation(dev) / test splits; only examples with a support passage" if USE_SCIQ else "not used"},
    "split_sizes": split_sizes,
    "preprocessing": "strip whitespace; pairs = (question, containing passage); corpus for evaluation = unique passages of each test set",
    "retrieval_threshold": round(float(THRESHOLD), 4),
    "retrieval_results": results,
    "threshold_results": results_threshold,
    "versions": {"python": platform.python_version(), "torch": torch.__version__,
                 "transformers": transformers.__version__, "sentence_transformers": sentence_transformers.__version__,
                 "datasets": hf_datasets.__version__},
}
json.dump(report, open(f"{MODEL_OUT}/papermind_config.json", "w"), indent=2)      # read by the backend
json.dump(report, open(f"{OUT_DIR}/reports/training_report.json", "w"), indent=2)
df.to_csv(f"{OUT_DIR}/reports/retrieval_results.csv", index=False)

zip_path = shutil.make_archive("/content/papermind_colab_outputs", "zip", OUT_DIR)
print("Saved ->", zip_path, f"({os.path.getsize(zip_path)/1e6:.0f} MB)")

## 11. Quick sanity check of the saved model

In [ ]:
from sentence_transformers import util
m = SentenceTransformer(MODEL_OUT, device="cuda")
passages = ["We trained the network with the Adam optimizer and a learning rate of 0.001.",
            "A limitation of this study is the small sample size.",
            "The dataset contains 10,000 labelled leaf images."]
q = "Which optimizer was used for training?"
scores = util.cos_sim(m.encode(q), m.encode(passages))[0]
for p, s in sorted(zip(passages, scores.tolist()), key=lambda x: -x[1]): print(f"{s:.3f}  {p}")
print("\nThreshold used by the app:", round(float(THRESHOLD), 3))

## 12. Get the files out of Colab
Run **one** of the two cells below.

**Option A – direct download** (simplest). **Option B – Google Drive** (more reliable if the browser download fails).

Then unzip on your laptop and:
1. copy the folder **`papermind-minilm`** into `papermind-ai/backend/models/`
2. keep the folder **`reports`** for your CA2 write-up (tables, plots, `training_report.json`)

In [ ]:
# Option A: browser download
from google.colab import files
files.download("/content/papermind_colab_outputs.zip")

In [ ]:
# Option B: copy to Google Drive  (My Drive/PaperMind/)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/PaperMind", exist_ok=True)
shutil.copy("/content/papermind_colab_outputs.zip", "/content/drive/MyDrive/PaperMind/")
print("Copied to Drive: PaperMind/papermind_colab_outputs.zip")